In [1]:
'''
Name: Reclustering_ABC10xv3_part3.ipynb
Purpose: creating a reference single cell RNA seq. object for RCTD analysis on Xenium data
        - part 3: reclustering cells
Date created: 1/22/26
Last updated: 7/23/26
Author: Chloe Lucido
'''

'\nName: Reclustering_ABC10xv3_part3.ipynb\nPurpose: creating a reference single cell RNA seq. object for RCTD analysis on Xenium data\n        - part 3: reclustering cells\nDate created: 1/22/26\nLast updated: 7/23/26\nAuthor: Chloe Lucido\n'

In [2]:
# import required modules 
    # copied these over from part 1
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt

# prevents datatype error when saving as h5ad obj
anndata.settings.allow_write_nullable_strings = True

In [14]:
####### PATHS/VARIABLES ##########
RAW_DOWNSAMPLED_OBJ = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260714_raw_WMB_10xv3_downsampled_metadata.h5ad" 
LOG2_DOWNSAMPLED_OBJ = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260714_log2_WMB_10xv3_downsampled_metadata.h5ad"
OUTPUT_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/"
DATE = "20260723_" 

raw_output_path = Path(OUTPUT_DIR) / f"{DATE}raw_WMB_10xv3_FINAL.h5ad"
log2_output_path = Path(OUTPUT_DIR) / f"{DATE}log2_WMB_10xv3_FINAL.h5ad"

In [4]:
# LOAD IN DOWNSAMPLED, MERGED OBJECT WITH CLUSTER METADATA ATTACHED 
raw_adata = anndata.read_h5ad(RAW_DOWNSAMPLED_OBJ)
log2_adata = anndata.read_h5ad(LOG2_DOWNSAMPLED_OBJ)

# inspect the objs 

print("Raw downsampled object: \n", raw_adata.obs.head(), "\n")

print("Log2 downsampled object: \n", log2_adata.obs.head())


# list clusters from object metadata
raw_adata.obs["cluster"]


Raw downsampled object: 
                                                         cell_barcode  \
cell_label                                                             
CACATGATCCCGTAAA-515_A03-WMB-10Xv3-Isocortex-1-...  CACATGATCCCGTAAA   
CAGCCAGGTACTGCCG-443_B03-WMB-10Xv3-Isocortex-1-...  CAGCCAGGTACTGCCG   
TAGTGCATCTACTGCC-390_B02-WMB-10Xv3-Isocortex-1-...  TAGTGCATCTACTGCC   
CCACCATAGATCACCT-308_A03-WMB-10Xv3-CTXsp-raw.h5ad   CCACCATAGATCACCT   
CTGCATCGTCCAGCGT-341_B01-WMB-10Xv3-CTXsp-raw.h5ad   CTGCATCGTCCAGCGT   

                                                         library_label  \
cell_label                                                               
CACATGATCCCGTAAA-515_A03-WMB-10Xv3-Isocortex-1-...  L8TX_210204_01_G05   
CAGCCAGGTACTGCCG-443_B03-WMB-10Xv3-Isocortex-1-...  L8TX_201204_01_D01   
TAGTGCATCTACTGCC-390_B02-WMB-10Xv3-Isocortex-1-...  L8TX_201015_01_H02   
CCACCATAGATCACCT-308_A03-WMB-10Xv3-CTXsp-raw.h5ad   L8TX_200730_01_B02   
CTGCATCGTCCAGCGT-341_B01-

cell_label
CACATGATCCCGTAAA-515_A03-WMB-10Xv3-Isocortex-1-raw.h5ad    0003 CLA-EPd-CTX Car3 Glut_1
CAGCCAGGTACTGCCG-443_B03-WMB-10Xv3-Isocortex-1-raw.h5ad    0003 CLA-EPd-CTX Car3 Glut_1
TAGTGCATCTACTGCC-390_B02-WMB-10Xv3-Isocortex-1-raw.h5ad    0004 CLA-EPd-CTX Car3 Glut_1
CCACCATAGATCACCT-308_A03-WMB-10Xv3-CTXsp-raw.h5ad          0001 CLA-EPd-CTX Car3 Glut_1
CTGCATCGTCCAGCGT-341_B01-WMB-10Xv3-CTXsp-raw.h5ad          0001 CLA-EPd-CTX Car3 Glut_1
                                                                       ...             
TGGGAGATCCTCACGT-398_D01-WMB-10Xv3-HPF-raw.h5ad                       5322 T cells NN_4
TGTGATGCAGTGTGCC-412_A07-WMB-10Xv3-HPF-raw.h5ad                       5322 T cells NN_4
TTCTAACAGGACAGCT-403_C06-WMB-10Xv3-HPF-raw.h5ad                       5322 T cells NN_4
TTGGATGCAGCCGGTT-409_C04-WMB-10Xv3-HPF-raw.h5ad                       5322 T cells NN_4
GTCATCCGTCACTCGG-398_D01-WMB-10Xv3-HPF-raw.h5ad                       5322 T cells NN_4
Name: cluster, Length

In [5]:
# assigning our own broad cluster and subcluster hierarchy

# create empty metadata columns 
raw_adata.obs["Broad_Cluster"] = pd.NA
log2_adata.obs["Broad_Cluster"] = pd.NA

# create empty metadata columns 
raw_adata.obs["Subcluster"] = pd.NA
log2_adata.obs["Subcluster"] = pd.NA

# CREATE DICTIONARY FOR BROAD CLUSTERS (how you want the broad clusters organized)

Broad_Cluster_map_by_supertype = {
    "NK Cells" : ["1200 NK cells NN_3"], 
    "T Cells" : ["1201 T cells NN_4"]
}


Broad_Cluster_map_by_subclass = {
    "Microglia" : ["334 Microglia NN"], 
    "BAM" : ["335 BAM NN"], 
    "Monocytes" : ["336 Monocytes NN"], 
    "DC" : ["337 DC NN"], 
    "ABC" : ["329 ABC NN"], 
    "VLMC" : ["330 VLMC NN"], 
    "Pericytes" : ["331 Peri NN"], 
    "SMC" : ["332 SMC NN"], 
    "Endothelial Cells" : ["333 Endo NN"], 
    "OPC" : ["326 OPC NN"], 
    "Oligodendrocytes" : ["327 Oligo NN"], 
    "Astrocytes" : ["317 Astro-CB NN", "318 Astro-NT NN", "319 Astro-TE NN", "320 Astro-OLF NN"], 
    "Astroependymal" : ["321 Astroependymal NN"], 
    "Tanycytes" : ["322 Tanycyte NN"], 
    "Ependymal" : ["323 Ependymal NN"], 
    "Hypendymal" : ["324 Hypendymal NN"], 
    "CP" : ["325 CHOR NN"]
}


Broad_Cluster_map_by_class = {
    "Dopaminergic_Neurons" : ["21 MB Dopa"],
    "Serotonergic_Neurons" : ["22 MB-HB Sero"],
    "GABAergic_Neurons" : ["05 OB-IMN GABA", "06 CTX-CGE GABA", "07 CTX-MGE GABA", "08 CNU-MGE GABA", "09 CNU-LGE GABA", "10 LSX GABA", "11 CNU-HYa GABA", "12 HY GABA", "20 MB GABA", "26 P GABA", "27 MY GABA", "28 CB GABA"],
    "Glutamatergic_Neurons" : ["01 IT-ET Glut", "02 NP-CT-L6b Glut", "03 OB-CR Glut", "04 DG-IMN Glut", "13 CNU-HYa Glut", "14 HY Glut", "15 HY Gnrh1 Glut", "16 HY MM Glut", "17 MH-LH Glut", "18 TH Glut", "19 MB Glut", "23 P Glut", "24 MY Glut", "25 Pineal Glut", "29 CB Glut"]
}


# CREATE DICTIONARY FOR SUBCLUSTERS 


Subcluster_map_by_supertype = {
    "NK Cells" : ["1200 NK cells NN_3"], 
    "T Cells" : ["1201 T cells NN_4"]
}

Subcluster_map_by_subclass = {
    "Microglia" : ["334 Microglia NN"], 
    "BAM" : ["335 BAM NN"], 
    "Monocytes" : ["336 Monocytes NN"], 
    "DC" : ["337 DC NN"], 
    "ABC" : ["329 ABC NN"], 
    "VLMC" : ["330 VLMC NN"], 
    "Pericytes" : ["331 Peri NN"], 
    "SMC" : ["332 SMC NN"], 
    "Endothelial Cells" : ["333 Endo NN"], 
    "OPC" : ["326 OPC NN"], 
    "Oligodendrocytes" : ["327 Oligo NN"], 
    "Astrocytes" : ["317 Astro-CB NN", "318 Astro-NT NN", "319 Astro-TE NN", "320 Astro-OLF NN"], 
    "Astroependymal" : ["321 Astroependymal NN"], 
    "Tanycytes" : ["322 Tanycyte NN"], 
    "Ependymal" : ["323 Ependymal NN"], 
    "Hypendymal" : ["324 Hypendymal NN"], 
    "CP" : ["325 CHOR NN"]
}

Subcluster_map_by_class = {
    "Dopaminergic_Neurons" : ["21 MB Dopa"],
    "Serotonergic_Neurons" : ["22 MB-HB Sero"]
}

GABA_Subcluster_map_by_subclass = {
    "LAMP5" : ["049 Lamp5 Gaba", "050 Lamp5 Lhx6 Gaba"], 
    "PVALB" : ["051 Pvalb chandelier Gaba", "052 Pvalb Gaba"], 
    "SNCG" : ["047 Sncg Gaba"], 
    "SST" : ["053 Sst Gaba", "056 Sst Chodl Gaba"], 
    "VIP" : ["046 Vip Gaba"]
}

Glut_Subcluster_map_by_subclass = {
    "L2_3_IT" : ["007 L2/3 IT CTX Glut", "008 L2/3 IT ENT Glut", "009 L2/3 IT PIR-ENTl Glut", "019 L2/3 IT PPP Glut", "020 L2/3 IT RSP Glut"], 
    "L5_IT" : ["005 L5 IT CTX Glut"], 
    "L6_IT" : ["004 L6 IT CTX Glut"], 
    "L5_ET" : ["022 L5 ET CTX Glut"], 
    "L6_CT" : ["028 L6b/CT ENT Glut", "029 L6b CTX Glut", "030 L6 CT CTX Glut"]
}



In [6]:
# adding map to object 


'''
Function: used to map predefined "dictionaries" from above to the downsampled 10xv3 object metadata
Input: metadata row (must be a pandas df) that has ABC cluster metadata information attached 
    (contains columns 'class', 'subclass', and 'cluster'"
Output: labels the pre-defined broad cluster that the cell belongs to in the 'Broad_Cluster' column
'''

def map_broad_cluster(row):
    
    # Mapping by class first 
    # .items() returns broad_cluster (broad) and class (class_list) pairs from the 'Broad_Cluster_map_by_class' df
    for broad, class_list in Broad_Cluster_map_by_class.items():

        # for loop goes through each class in the class_list belonging to the broad_cluster value
        for cls in class_list:

            # if the class matches the class listed in the 'class' column from the downsampled object, the 'broad_cluster' value will be added to the 'Broad_Cluster' column
            if cls in row['class']:
                return broad
    
    # if couldn't find a match for class, then use Broad_Cluster_map_by_subclass
    for broad, subclass_list in Broad_Cluster_map_by_subclass.items():

        # for loop goes through each subclass in the subclass_list belonging to the broad_cluster value
        for subcls in subclass_list:
            
            # if the subclass matches, the 'broad_cluster' value will be added 
            if subcls in row['subclass']:
                return broad

    # if couldn't find a match for subclass, then use Broad_Cluster_map_by_supertype
    for broad, supertype_list in Broad_Cluster_map_by_supertype.items():

        # for loop goes through each supertype in the supertype_list belonging to the broad_cluster value
        for suptype in supertype_list:
            
            # if the supertype matches, the 'broad_cluster' value will be added 
            if suptype in row['supertype']:
                return broad
                
    
    # if nothing matches
    return "Unknown"

def map_subcluster(row):

    # for neuronal subclusters 
    if row['Broad_Cluster'] == "Glutamatergic_Neurons":
        for sub, subclass_list in Glut_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None 
    
    if row['Broad_Cluster'] == "GABAergic_Neurons":
        for sub, subclass_list in GABA_Subcluster_map_by_subclass.items():
            for subcls in subclass_list:
                if subcls in row['subclass']:
                    return sub
                    
        return None    

    
    # other subclusters  

    # mapping by class dictionary
    for sub, class_list in Subcluster_map_by_class.items():
        for cls in class_list:
            if cls in row['class']:
                return sub
                
    # mapping by subclass dictionary 
    for sub, subclass_list in Subcluster_map_by_subclass.items():
        for s in subclass_list:
            if s in row['subclass']:
                return sub
                
    # mapping by supertype dictionary  
    for sub, supertype_list in Subcluster_map_by_supertype.items():
        for su in supertype_list:
            if su in row['supertype']:
                return sub

    
    return None




In [7]:
# extracting metadata from downsampled object 
raw_meta = raw_adata.obs.copy()
log2_meta = log2_adata.obs.copy()


# applying the function to the metadata object 
raw_meta['Broad_Cluster'] = raw_meta.apply(map_broad_cluster, axis = 1)
log2_meta['Broad_Cluster'] = log2_meta.apply(map_broad_cluster, axis = 1)

raw_meta['Subcluster'] = raw_meta.apply(map_subcluster, axis = 1)
log2_meta['Subcluster'] = log2_meta.apply(map_subcluster, axis = 1)

# dropping cells that were not assigned to a broad cluster 
'''
In this case, there are no unannotated cells present (we removed them in part 2). 
The cells in the unknown cluster are cells belonging to classes that we did not specify in the dictionary 
bc we do not need them for ARIA project (i.e. OEC, B cells, etc.) and we don't have any markers for them in our gene panel.
'''
raw_meta = raw_meta[raw_meta['Broad_Cluster'] != "Unknown"].copy()
log2_meta = log2_meta[log2_meta['Broad_Cluster'] != "Unknown"].copy()


# assign back to the downsampled obj
raw_adata = raw_adata[raw_meta.index, :].copy()
raw_adata.obs = raw_meta

log2_adata = log2_adata[log2_meta.index, :].copy()
log2_adata.obs = log2_meta


In [8]:
print("Raw downsampled obj: \n", raw_adata.n_obs)
print(raw_adata.obs["Broad_Cluster"].value_counts())

print("log2 downsampled obj: \n", log2_adata.n_obs)
print(log2_adata.obs["Subcluster"].value_counts())

Raw downsampled obj: 
 250825
Broad_Cluster
GABAergic_Neurons        64995
Oligodendrocytes         58259
Astrocytes               43877
Endothelial Cells        16174
OPC                      14814
SMC                       9009
Microglia                 8554
VLMC                      8544
Glutamatergic_Neurons     8133
Pericytes                 5611
BAM                       5516
Ependymal                 3096
Astroependymal            1174
Tanycytes                 1004
ABC                        824
CP                         447
DC                         261
T Cells                    165
NK Cells                   128
Hypendymal                 111
Dopaminergic_Neurons        50
Serotonergic_Neurons        50
Monocytes                   29
Name: count, dtype: int64
log2 downsampled obj: 
 250825
Subcluster
Oligodendrocytes        58259
Astrocytes              43877
Endothelial Cells       16174
OPC                     14814
SMC                      9009
Microglia                

In [12]:
# SANITY CHECK: check for NAs in cluster columns
print("Raw adata obj: ")

print(raw_adata.obs["Broad_Cluster"].isna().sum())
# output: 0

print(raw_adata.obs["Subcluster"].isna().sum())
# output: 63327

print("Log2 obj: \n")

print(log2_adata.obs["Broad_Cluster"].isna().sum())
# output: 0

print(log2_adata.obs["Subcluster"].isna().sum())
# output: 63327

# verify number of cells/obs before taking out NAs
print("Raw adata total number of cells before cleaning: ", raw_adata.n_obs)
print("log2 adata total number of cells before cleaning: ", log2_adata.n_obs)

# subset out cells with no Subcluster annotation (have NA)
clean_raw_adata = raw_adata[raw_adata.obs["Subcluster"].notna()].copy()
clean_log2_adata = log2_adata[log2_adata.obs["Subcluster"].notna()].copy()

# verify number of cells/obs after taking out NAs
print("Raw adata total number of cells after cleaning: ", clean_raw_adata.n_obs)
print("log2 adata total number of cells after cleaning: ", clean_log2_adata.n_obs)


Raw adata obj: 
0
63327
Log2 obj: 

0
63327
Raw adata total number of cells before cleaning:  250825
log2 adata total number of cells before cleaning:  250825
Raw adata total number of cells after cleaning:  187498
log2 adata total number of cells after cleaning:  187498


In [15]:
# save downsampled object with cluster metadata and our own cluster levels added in
clean_raw_adata.write_h5ad(raw_output_path)
clean_log2_adata.write_h5ad(log2_output_path)

